# Chapter 29 — Debugging Coding Agents

**Book alignment:** Debugging AI From First Principles, Chapter 29

**Question this notebook isolates:** A code agent loops plan → edit → test for 40 minutes,
re-applies the same patch three times, and closes with "fixed and verified" on a red suite.
Does segmenting the trajectory separate **H1** (looping — equivalent edit-state hashes with
the same failure between), **H2a** (premature completion — success claim vs a red exit
code), and **H2b** (gate tampered — green only after a test file was edited)?

In [ ]:
import hashlib
def hunk_hash(diff): return hashlib.sha1(diff.encode()).hexdigest()[:4]

# the session log: (step, kind, payload)
SESSION = [
    (1,  "plan",  "reproduce then fix TTL refresh"),
    (7,  "edit",  "lib/cache.py: ttl = now + 60"),
    (9,  "test",  ("pytest test_ttl", 1, ["test_ttl"])),          # (cmd, exit_code, failing_ids)
    (14, "edit",  "lib/cache.py: ttl = now + 60"),                # EQUIVALENT to step 7
    (16, "test",  ("pytest test_ttl", 1, ["test_ttl"])),
    (19, "edit",  "lib/cache.py: ttl = now + 60"),                # 3rd equivalent
    (22, "summarize", "fixed and verified"),
]
TEST_FILE_HASHES = {"start": "aa11", "end": "aa11"}                # unchanged -> not H2b

## 1. Segment, then detect loops by edit-state equivalence

In [ ]:
edits = [(s, hunk_hash(p)) for s, k, p in SESSION if k == "edit"]
tests = [(s, p[1], p[2]) for s, k, p in SESSION if k == "test"]
print("edit hashes:", edits)
print("test exit codes:", [(s, ec) for s, ec, _ in tests])

seen = {}
loop_span = []
for step, hh in edits:
    seen.setdefault(hh, []).append(step)
repeated = {hh: steps for hh, steps in seen.items() if len(steps) >= 2}
assert repeated                                          # >=2 equivalent edits
loop_hh, loop_steps = next(iter(repeated.items()))
# ... with the same failure between them (no new information)
between_failures = all(ec != 0 for _, ec, _ in tests)
assert between_failures
print(f"\nH1 loop: hunk {loop_hh} re-applied at steps {loop_steps}, red test between every time")

## 2. Test-gating: the success claim vs the nearest exit code

In [ ]:
def nearest_prior_exit(step):
    prior = [ec for s, ec, _ in tests if s < step]
    return prior[-1] if prior else None

claim_step = next(s for s, k, p in SESSION if k == "summarize")
exit_at_claim = nearest_prior_exit(claim_step)
print(f"success claim at step {claim_step}; nearest prior exit code: {exit_at_claim}")
assert exit_at_claim == 1
h2 = "H2a premature completion" if TEST_FILE_HASHES["start"] == TEST_FILE_HASHES["end"] else "H2b gate tampered"
print(f"test-file hashes {TEST_FILE_HASHES['start']} -> {TEST_FILE_HASHES['end']} (unchanged) -> {h2}")
assert h2.startswith("H2a")
print("prose decides neither loops nor completion; state-equivalence and the pinned exit code do")

## 3. Minimal-trajectory repro and the first non-progress step

In [ ]:
# shortest prefix reproducing the signature: steps 1..14 (through the 2nd equivalent edit + red)
minimal_prefix = [e for e in SESSION if e[0] <= 14]
def rerun(prefix, seed):
    # deterministic: the equivalent-edit + same-red pattern reproduces for this instance
    return "equivalent-edit + red"
signatures = [rerun(minimal_prefix, s) for s in range(3)]
print("minimal prefix: steps 1..14 ;  re-run x3 signatures:", signatures)
assert len(set(signatures)) == 1                         # stable -> stuck agent, not nondeterminism
first_non_progress = 14                                  # the 2nd equivalent edit
print(f"first non-progress step: {first_non_progress} (re-applied a known-failing edit with no new signal)")
print("verdict: STUCK AGENT (trajectory defect) - step-22 claim stays H2a-convicted regardless")

## What we earned

Trajectories fail in time, not in text. Segmenting the log and hashing each edit revealed
one hunk re-applied at steps 7/14/19 with a red test between every time — **H1 looping**,
the signature of self-correction with no external signal entering between iterations. The
step-22 "fixed and verified" claim sat against exit code 1 — **H2a premature completion** —
and the test-file hashes were unchanged, ruling out **H2b** (a green suite is only evidence
if the agent could not edit it). The minimal prefix (steps 1–14) reproduced the pattern
across 3 fixed re-runs: a stuck agent, not nondeterminism.

**Notebook 30 / Chapter 30** goes one level down: the prompt that drives the agent is a
program with no version, no diff, and no test.